# A/B Testing: Data Cleaning and Preparation

This notebook focuses on cleaning and validating the raw A/B testing dataset. Ensuring data integrity is a critical first step before performing any statistical analysis or hypothesis testing.

Our goals for this notebook are:
1. Load and inspect the raw data.
2. Handle missing values and duplicates.
3. Ensure users are uniquely assigned to a single group.
4. Correct data types.
5. Verify experimental integrity (e.g., control group sees the old page, treatment group sees the new page).
6. Export the clean dataset for exploratory data analysis (EDA) and hypothesis testing.

---

## 1. Import Necessary Libraries

First, we will import the required Python libraries for data manipulation.

In [1]:
import pandas as pd
import numpy as np

# Set display options for pandas to ensure we can see all columns
pd.set_option('display.max_columns', None)

---

## 2. Load the Dataset

We will load the raw A/B testing data from our `data/raw/` directory.

In [5]:
# Define file paths
raw_data_path = (r'../data/raw/ab_testing.csv')
processed_data_path = (r'../data/processed/cleaned_data.csv')

# Load the dataset
df = pd.read_csv(raw_data_path)
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")


Dataset loaded successfully with 294478 rows and 12 columns.


---

## 3. Initial Data Inspection

Let's take a quick look at the dataset to understand its structure, features, and overall summary statistics.

In [6]:
# Preview the first 5 rows
display(df.head())

# Check column data types and non-null counts
print("\n--- DataFrame Info ---")
df.info()

# Summary statistics for numerical and categorical columns
print("\n--- Summary Statistics ---")
display(df.describe(include='all'))

,user_id,timestamp,group,landing_page,converted,age,gender,location,session_duration,pages_visited,device_type,purchase_amount
0,U1,2025-08-02 15:27:54.137058,control,old_page,0,37,Male,Pakistan,3.69,4,Mobile,0.0
1,U2,2024-04-22 10:22:51.712050,treatment,new_page,0,31,Female,UK,1.29,3,Desktop,0.0
2,U3,2024-08-14 21:35:11.135894,treatment,new_page,0,38,Male,US,3.72,5,Desktop,0.0
3,U4,2025-03-19 03:28:51.120807,treatment,new_page,0,28,Female,India,7.76,2,Mobile,0.0
4,U5,2024-12-22 13:13:17.973162,control,old_page,0,33,Male,Australia,6.78,6,Mobile,0.0



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           294478 non-null  object 
 1   timestamp         294478 non-null  object 
 2   group             294478 non-null  object 
 3   landing_page      294478 non-null  object 
 4   converted         294478 non-null  int64  
 5   age               294478 non-null  int64  
 6   gender            294478 non-null  object 
 7   location          294478 non-null  object 
 8   session_duration  294478 non-null  float64
 9   pages_visited     294478 non-null  int64  
 10  device_type       294478 non-null  object 
 11  purchase_amount   294478 non-null  float64
dtypes: float64(2), int64(3), object(7)
memory usage: 27.0+ MB

--- Summary Statistics ---


,user_id,timestamp,group,landing_page,converted,age,gender,location,session_duration,pages_visited,device_type,purchase_amount
count,294478,294478,294478,294478,294478.000000,294478.000000,294478,294478,294478.000000,294478.000000,294478,294478.000000
unique,294478,294478,2,2,NaN,NaN,3,7,NaN,NaN,3,NaN
top,U1,2025-08-02 15:27:54.137058,treatment,new_page,NaN,NaN,Male,US,NaN,NaN,Desktop,NaN
freq,1,1,147552,147552,NaN,NaN,144708,88342,NaN,NaN,176692,NaN
mean,NaN,NaN,NaN,NaN,0.149172,31.902084,NaN,NaN,5.002136,4.019550,NaN,5.610490
std,NaN,NaN,NaN,NaN,0.356259,9.250043,NaN,NaN,1.980285,1.965182,NaN,15.438006
min,NaN,NaN,NaN,NaN,0.000000,18.000000,NaN,NaN,0.500000,1.000000,NaN,0.000000
25%,NaN,NaN,NaN,NaN,0.000000,25.000000,NaN,NaN,3.640000,3.000000,NaN,0.000000
50%,NaN,NaN,NaN,NaN,0.000000,32.000000,NaN,NaN,4.990000,4.000000,NaN,0.000000
75%,NaN,NaN,NaN,NaN,0.000000,38.000000,NaN,NaN,6.340000,5.000000,NaN,0.000000


---

## 4. Check for and Handle Missing Values

Missing data can skew our A/B test results. We need to identify any nulls and handle them appropriately.

In [7]:
# Check for missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values[missing_values > 0])

# Strategy: 
# If critical core metrics (user_id, group, landing_page, converted) are missing, we drop them.
# For other demographic/behavioral columns, we drop them to maintain a clean baseline for the test.
# (Note: In a real-world scenario, you might choose to impute values like 'age' or 'purchase_amount')

initial_rows = df.shape[0]
df = df.dropna()
dropped_rows = initial_rows - df.shape[0]

print(f"\nDropped {dropped_rows} rows due to missing values. Current shape: {df.shape}")

Missing values per column:
 Series([], dtype: int64)

Dropped 0 rows due to missing values. Current shape: (294478, 12)


---

## 5. Check for and Remove Duplicates

In standard A/B testing, a user should only be counted once. If a user is exposed to both the control and treatment groups, they are contaminated and must be removed to preserve the validity of the test.

In [8]:
# 5.1 Drop exact duplicate rows
df = df.drop_duplicates()

# 5.2 Handle duplicate user_ids (cross-contamination)
# Find users who appear more than once in the dataset
session_counts = df['user_id'].value_counts()
multi_session_users = session_counts[session_counts > 1].index

print(f"Number of users with multiple sessions/entries: {len(multi_session_users)}")

# Drop these users completely to avoid cross-contamination between test groups
df = df[~df['user_id'].isin(multi_session_users)]

print(f"Shape after removing contaminated users: {df.shape}")
# Verify user_id is now strictly unique
assert df['user_id'].nunique() == df.shape[0], "Error: Duplicate user_ids still exist!"

Number of users with multiple sessions/entries: 0
Shape after removing contaminated users: (294478, 12)


---

## 6. Verify and Correct Data Types

We need to ensure all columns are cast to their appropriate mathematical data types for later statistical analysis.

In [9]:
# Convert timestamp to datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Ensure strict integer types for counts and binary flags
df['converted'] = df['converted'].astype(int)
df['age'] = df['age'].astype(int)
df['pages_visited'] = df['pages_visited'].astype(int)

# Ensure float types for continuous variables
df['session_duration'] = df['session_duration'].astype(float)
df['purchase_amount'] = df['purchase_amount'].astype(float)

# Check the updated data types
print(df.dtypes)

user_id                     object
timestamp           datetime64[ns]
group                       object
landing_page                object
converted                    int64
age                          int64
gender                      object
location                    object
session_duration           float64
pages_visited                int64
device_type                 object
purchase_amount            float64
dtype: object


---

## 7. Ensure Data Integrity (Experimental Setup)

A crucial check in A/B testing is ensuring the tracking was implemented correctly. 
* The `control` group should *only* see the `old_page`.
* The `treatment` group should *only* see the `new_page`.

Any mismatches indicate logging errors and must be removed.

In [10]:
# Check how many mismatches we have
mismatches = df[((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')) | \
                ((df['group'] == 'control') & (df['landing_page'] == 'new_page'))]

print(f"Number of mismatched rows detected: {len(mismatches)}")

# Filter the dataframe to only include correct group/page assignments
df = df[((df['group'] == 'control') & (df['landing_page'] == 'old_page')) | \
        ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))]

print(f"Shape after removing mismatched rows: {df.shape}")

# Double check the mapping
pd.crosstab(df['group'], df['landing_page'])

Number of mismatched rows detected: 0
Shape after removing mismatched rows: (294478, 12)


landing_page,new_page,old_page
group,,
control,0,146926
treatment,147552,0


---

## 8. Export the Cleaned Dataset

The data is now clean, deduplicated, correctly typed, and experimentally valid. We will export this to the `processed` folder so it can be picked up by the next notebook for EDA and statistical testing.

In [11]:
# Save the cleaned dataframe to a CSV file
df.to_csv(processed_data_path, index=False)

print(f"Cleaned dataset successfully exported to: {processed_data_path}")

Cleaned dataset successfully exported to: ../data/processed/cleaned_data.csv
